# Native SAM3 sunglasses detection (Kaggle)

Counts pairs of sunglasses displayed on a retail rack, **excluding pairs worn on a person**. Uses the standalone `sam3-verbose-counting/` pipeline.


## 1. Setup & checkpoint

Clone the repo, install only the SAM3 deps missing from the kernel, and download the gated `sam3.pt`.


In [ ]:
import importlib, importlib.util, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/fez-Ox/pxModel-Object-Counting.git"  # <-- edit me
REPO_DIR = Path("/kaggle/working/pxModel-localization")
SAM3_APP = REPO_DIR / "sam3-verbose-counting"

if not (REPO_DIR / ".git").is_dir():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    branch = subprocess.run(
        ["git", "-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{branch}"], check=True)

# Install only the SAM3 deps missing from the Kaggle kernel
missing = [m for m in ["torch", "torchvision", "PIL", "numpy", "timm", "einops",
                       "ftfy", "regex", "wrapt", "typing_extensions"]
           if importlib.util.find_spec(m) is None]
missing = ["Pillow" if m == "PIL" else m for m in missing]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=True)

sys.path.insert(0, str(SAM3_APP))
# Drop stale cached modules/bytecode so the just-synced code is used
for _m in list(sys.modules):
    if _m in ("infer", "download_model") or _m == "detectors" or _m.startswith("sam3"):
        del sys.modules[_m]
for _c in SAM3_APP.rglob("__pycache__"):
    shutil.rmtree(_c, ignore_errors=True)
importlib.invalidate_caches()

from infer import build_counter, annotate
from detectors import get_detector, DetectionOptions
import download_model as sam3_download

sam3_path = sam3_download.download_model(
    url=sam3_download.DEFAULT_URL,
    output=sam3_download.DEFAULT_OUTPUT,
    force=False, timeout=180, token=None,  # auto-detects HF_TOKEN / Kaggle secret
)
print("Checkpoint ready:", sam3_path)


## 2. Run sunglasses detection

Build the persistent counter, run the decoupled `sunglasses` detection task, and show the annotated result (pairs worn on people are dropped).


In [ ]:
import torch
from IPython.display import display

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

counter = build_counter(threshold=0.5)      # loads sam3.pt on cuda (or cpu)
sunglasses = get_detector("sunglasses")   # decoupled detection task

image_path = "/kaggle/input/my-dataset/rack_photo.jpg"  # <-- edit me
prompt = "all pairs of black sunglasses displayed on the retail rack"

result = sunglasses.run(counter, image_path, DetectionOptions(prompt=prompt))

print(f"Count (worn pairs excluded): {result['count']}")
if "raw_count" in result:
    print(f"  model boxes: {result['raw_count']} | after cleanup: "
          f"{result['deduplicated_count']} | redundant removed: {result['redundant_box_count']}")
if "filtered_count" in result:
    print(f"  removed (worn / on a person): {result['filtered_count']}")

display(annotate(image_path, prompt, result["boxes"], result["scores"]))


## 3. Cleanup

Release the model from GPU memory.


In [ ]:
del counter, sunglasses, result
import torch, gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Released SAM3.")
